# Missing Meta Link Analysis

In [ ]:
from bddl.knowledge_base import *

## Analysis by Object

In [ ]:
objects_w_missing_metadata = {
    o: o.missing_meta_links for o in Object.all_objects() if o.missing_meta_links
}

In [ ]:
len(objects_w_missing_metadata)

In [ ]:
import collections

by_type = collections.Counter(
    mt for missing_types in objects_w_missing_metadata.values() for mt in missing_types
)
print(by_type.most_common())

### Split by provider for work

In [ ]:
todo = {"togglebutton", "slicer", "heatsource", "fluidsource", "fluidsink"}
by_provider = collections.defaultdict(dict)
for o, mts in objects_w_missing_metadata.items():
    required_mts = todo.intersection(mts)
    if not required_mts:
        continue

    by_provider[o.provider][o.name] = required_mts

for provider, objects in sorted(by_provider.items()):
    print(provider)
    for name, mts in objects.items():
        print(f"  {name}: {mts}")
    print()

## Analysis by Synset

In [ ]:
synsets_w_missing_metadata = [
    s for s in Synset.all_objects() if not s.has_fully_supporting_object
]
print(len(synsets_w_missing_metadata))

In [ ]:
tr_synsets_w_missing_metadata = [
    s
    for s in Synset.all_objects()
    if len(s.tasks) > 0 and not s.has_fully_supporting_object
]
print(len(tr_synsets_w_missing_metadata))
print(tr_synsets_w_missing_metadata)

In [ ]:
# Check if any sliceable task relevant synsets are bad
def get_sliced(s):
    return Synset.get("half__" + s.name.split(".n.")[0] + ".n.01")


transition_relevant_synsets = {
    s
    for t in Task.all_objects()
    for transition in t.relevant_transitions
    for s in list(transition.output_synsets) + list(transition.input_synsets)
}

sliceable_tr_synsets_w_missing_metadata = [
    (s, get_sliced(s))
    for s in Synset.all_objects()
    if (
        any(p.name == "sliceable" for p in s.properties)
        and not any(
            ml.name == "subpart" for o in s.matching_objects for ml in o.meta_links
        )
    )
    and (len(get_sliced(s).tasks) > 0 or s in transition_relevant_synsets)
]
print(
    len(sliceable_tr_synsets_w_missing_metadata),
    "missing subpart annotation despite having sliceable part",
)
print(
    "\n".join(
        f"{s.name} -> {sliced.name}"
        for s, sliced in sliceable_tr_synsets_w_missing_metadata
    )
)
print()
missing_half_objs = [
    sliced
    for s, sliced in sliceable_tr_synsets_w_missing_metadata
    if len(sliced.matching_objects) == 0
]
print(len(missing_half_objs), "missing half objs")
print(missing_half_objs)

In [ ]:
clam = Synset.get("clam.n.03")
for task in Task.all_objects():
    for transition in task.relevant_transitions:
        if clam in transition.input_synsets or clam in transition.output_synsets:
            print(
                task.name,
                transition.name,
                transition.input_synsets,
                transition.output_synsets,
            )

In [ ]:
# Check if any sliceable task relevant synsets are bad
sliceable_synsets_w_missing_metadata = [
    (s, get_sliced(s))
    for s in Synset.all_objects()
    if (
        any(p.name == "sliceable" for p in s.properties)
        and len(s.matching_objects) > 0
        and not any(
            ml.name == "subpart" for o in s.matching_objects for ml in o.meta_links
        )
    )
]
print(
    len(sliceable_synsets_w_missing_metadata),
    "missing subpart annotation despite having sliceable part",
)
print(
    "\n".join(
        f"{s.name} -> {sliced.name}"
        for s, sliced in sliceable_synsets_w_missing_metadata
    )
)
print()
missing_half_objs = [
    sliced
    for s, sliced in sliceable_synsets_w_missing_metadata
    if len(sliced.matching_objects) == 0
]
print(len(missing_half_objs), "missing half objs")
print(missing_half_objs)

In [ ]:
# What objects are fillable?
fillable_objs = [
    o
    for o in Object.all_objects()
    if any(p.name == "fillable" for p in o.category.synset.properties)
]
print(len(fillable_objs))
print(sorted([x.name for x in fillable_objs]))

In [ ]:
# Check what synsets don't have full versions
